In [2]:
# pip install strip-markdown
# Optional: pip install requests is not needed; this notebook uses the Python standard library for Ollama API calls.

In [4]:
# pip install pydantic
# Optional: keep or remove these comments as you customize the notebook.

In [1]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Sequence
import json
import urllib.request
import urllib.error

@dataclass
class ChatMessage:
    role: str
    content: str

@dataclass
class OllamaResponse:
    content: str
    raw: dict[str, Any]

class OllamaChat:
    def __init__(
        self,
        model: str = "llama3.2:latest",
        base_url: str = "http://localhost:11434",
        temperature: float = 0.7,
        timeout: int = 300,
    ):
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.temperature = temperature
        self.timeout = timeout

    def invoke(self, messages: Sequence[ChatMessage | dict[str, str]]) -> OllamaResponse:
        payload_messages = []
        for msg in messages:
            if isinstance(msg, ChatMessage):
                payload_messages.append({"role": msg.role, "content": msg.content})
            else:
                payload_messages.append({"role": msg["role"], "content": msg["content"]})

        payload = {
            "model": self.model,
            "messages": payload_messages,
            "options": {"temperature": self.temperature},
            "stream": False,
        }

        request = urllib.request.Request(
            f"{self.base_url}/api/chat",
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            method="POST",
        )

        try:
            with urllib.request.urlopen(request, timeout=self.timeout) as response:
                data = json.loads(response.read().decode("utf-8"))
        except urllib.error.URLError as exc:
            raise RuntimeError(
                f"Could not reach Ollama at {self.base_url}. "
                "Make sure Ollama is running and the model is available."
            ) from exc

        return OllamaResponse(
            content=data["message"]["content"],
            raw=data,
        )

def build_user_prompt(
    task: str,
    somecode: str = "",
    extra_context: str = "",
    notes: str = "",
) -> str:
    template = """
Task:
{task}

Code:
{somecode}

Extra context:
{extra_context}

Notes:
{notes}

Please answer clearly and include security concerns if relevant.
"""
    return template.format(
        task=task.strip(),
        somecode=somecode.strip(),
        extra_context=extra_context.strip(),
        notes=notes.strip(),
    )

In [2]:
chat_model = OllamaChat(
    model="llama3.2:latest",
    base_url="http://localhost:11434",
    temperature=0.7,
)

In [3]:
task = "Analyse the code below and explain what it does. Then point out any security issues and explain why they matter."
extra_context = "I want the explanation to be practical and beginner-friendly."
notes = "Focus on behavior, dependencies, inputs, outputs, and any risky patterns."

In [5]:
with open("data/Code.txt", "r", encoding="utf-8") as file:
    code = file.read()

# print(code)

formatted_prompt = build_user_prompt(
    task=task,
    somecode=code,
    extra_context=extra_context,
    notes=notes,
)

messages = [
    ChatMessage(role="system", content="You are a careful coding assistant."),
    ChatMessage(role="user", content=formatted_prompt),
]

response = chat_model.invoke(messages)

In [7]:
import strip_markdown

output = strip_markdown.strip_markdown(response.content)
# output_raw=(response.content)
# print(output_raw)
print(output)

Code Analysis
This code is a simple password manager that allows users to store and retrieve passwords. It has three main options:

Store new password
View stored passwords
Exit the program

Here's a step-by-step breakdown of what the code does:

The program opens a file named "definitely_not_a_file_full_of_passwords.txt" in append mode.
It enters a loop where it displays the menu options and waits for the user to input their choice.
If the user chooses option 1 (Store new password), the program prompts them to create a username, password, confirm password, and website URL.
The program checks if the passwords match, is at least 7 characters long, and if the username already exists in the file. If any of these conditions are not met, it displays an error message and asks the user to restart.
If all checks pass, the program appends the new password information to the file.
If the user chooses option 2 (View stored passwords), the program reads the entire file, splits each line into usern